In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow.keras as keras
import warnings
warnings.filterwarnings('ignore')

### Training MLP Model

In [2]:
# The number of seconds to use for each audio clip when extracting features and training the model. Either 3 or 30.
NUMBER_OF_SECONDS = 3
# Columns not taken into consideration for training the model, as they are not relevant for genre classification or are redundant.
COLUMNS_TO_DROP = ['filename', 'label', 'length', 'tempo','harmony_mean', 'harmony_var', 'perceptr_var', 'perceptr_mean']

In [3]:
data_path = f"../data/features_{NUMBER_OF_SECONDS}_sec.csv"

def load_data(data_path):
    df = pd.read_csv(data_path)

    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['label'])

    X = df.drop(columns=COLUMNS_TO_DROP).values
    y = df['label'].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    print(f"data loaded: {X.shape[0]} samples, {X.shape[1]} features.")
    return X, y, label_encoder, scaler  # Return scaler

X, y, encoder, scaler = load_data(data_path)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# Build MLP Topology
model = keras.Sequential([
    # Input layer: input_dim is the number of features in your CSV
    keras.layers.Dense(512, activation='relu', input_shape=(X.shape[1],)),
    keras.layers.Dropout(0.3), # Added dropout to prevent overfitting

    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(64, activation='relu'),

    # Output layer (10 genres)
    keras.layers.Dense(10, activation='softmax')
])

optimiser = keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimiser,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Train
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    batch_size=32,
                    epochs=50)

data loaded: 9990 samples, 52 features.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        27,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,562 (685.79 KB)

 Trainable params: 175,562 (685.79 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.3213 - loss: 1.9394 - val_accuracy: 0.4825 - val_loss: 1.5280
Epoch 2/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 997us/step - accuracy: 0.4868 - loss: 1.4419 - val_accuracy: 0.5686 - val_loss: 1.2359
Epoch 3/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 972us/step - accuracy: 0.5607 - loss: 1.2501 - val_accuracy: 0.6223 - val_loss: 1.1065
Epoch 4/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 968us/step - accuracy: 0.6050 - loss: 1.1368 - val_accuracy: 0.6500 - val_loss: 1.0249
Epoch 5/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 967us/step - accuracy: 0.6295 - loss: 1.0759 - val_accuracy: 0.6860 - val_loss: 0.9553
Epoch 6/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 955us/step - accuracy: 0.6574 - loss: 1.0042 - val_accuracy: 0.7107 - val_loss: 0.9003
Epoch 7/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 973us/step - accuracy: 0.6715 - loss: 0.9640 - val_accuracy: 0.7157 - val_loss: 0.8688
Epoch 8/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 999us/step - accuracy: 0.6790 - loss: 0.9182 - va

### Evaluating Model Using Custom Song

In [4]:
from src.extract_features import extract_features

NEW_SONG_PATH = "../data/genres_original/reggae/reggae.00027.wav"

raw_features_df = extract_features(NEW_SONG_PATH, segment_duration=NUMBER_OF_SECONDS)
raw_features_df

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,reggae.00027.wav,3.0,0.352092,0.109867,0.130797,0.004736,1784.115776,4.872008e+05,2227.136481,238695.607472,...,85.015686,-4.765120,64.310326,2.124596,59.804111,-10.804342,52.739101,-4.732891,77.199852,unknown
1,reggae.00027.wav,3.0,0.386090,0.099000,0.137611,0.005301,1885.877074,8.218617e+05,2183.801913,346768.065624,...,114.562172,-12.541590,108.803299,-0.482641,50.439800,-6.373384,89.506683,3.088569,68.818100,unknown
2,reggae.00027.wav,3.0,0.409195,0.091481,0.118830,0.006331,2801.719516,1.457331e+06,2531.149581,172517.098663,...,85.433273,-10.077676,68.775726,3.149647,46.381989,-1.288200,85.018906,5.926558,31.963154,unknown
3,reggae.00027.wav,3.0,0.389593,0.102011,0.128109,0.003596,1995.276551,5.847798e+05,2346.702062,213474.183621,...,76.537758,-6.336252,57.719551,0.165549,50.624611,-6.379256,37.024132,-1.992779,38.231159,unknown
4,reggae.00027.wav,3.0,0.430438,0.096207,0.113174,0.005213,2055.509791,6.522157e+05,2341.141814,203230.724958,...,74.307617,-3.542618,79.103951,4.057223,40.836590,1.951746,70.377106,1.807470,38.194057,unknown
5,reggae.00027.wav,3.0,0.401098,0.105039,0.128482,0.004317,2122.506051,7.591050e+05,2304.109715,198854.414755,...,70.452644,-8.193923,65.258461,1.426303,45.005733,-6.656531,42.765236,-0.595739,59.229752,unknown
6,reggae.00027.wav,3.0,0.443514,0.095210,0.119288,0.004922,2011.274800,6.009171e+05,2444.129134,216055.533346,...,69.012550,-5.132600,39.949318,4.461343,37.172192,1.783198,36.525375,5.057328,37.492382,unknown
7,reggae.00027.wav,3.0,0.380496,0.092000,0.128108,0.005503,2297.417491,6.102105e+05,2417.311357,201478.396762,...,62.712891,-11.933676,76.494003,0.674491,50.568111,-1.860376,98.178825,4.024657,53.233818,unknown
8,reggae.00027.wav,3.0,0.396696,0.103809,0.145057,0.005021,1834.243157,6.031630e+05,2122.998158,218999.746910,...,80.731255,-4.490624,60.994759,5.123927,97.087036,3.179155,84.648651,6.818520,46.862164,unknown
9,reggae.00027.wav,3.0,0.441811,0.092264,0.119072,0.005370,2110.256697,9.983919e+05,2305.968840,257511.102882,...,54.392044,-5.102010,88.207932,4.040416,30.695366,0.047705,37.738834,1.643048,29.620115,unknown


In [5]:
COLUMNS_TO_DROP.append('label')
raw_features_df = raw_features_df.drop(columns=COLUMNS_TO_DROP)
input_data = raw_features_df.values

scaled_input = scaler.transform(input_data)
predictions = model.predict(scaled_input)

# Average predictions across all segments
avg_predictions = np.mean(predictions, axis=0)

for i in range(len(avg_predictions)):
    print(f"{encoder.inverse_transform([i])[0]}: {avg_predictions[i]*100:.2f}%")
predicted_index = np.argmax(avg_predictions)

predicted_genre = encoder.inverse_transform([predicted_index])[0]

print(f"--- Prediction Results ---")
print(f"Predicted Genre: {predicted_genre.upper()}")
print(f"Confidence: {np.max(avg_predictions) * 100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
blues: 0.28%
classical: 0.00%
country: 0.02%
disco: 4.65%
hiphop: 2.41%
jazz: 0.00%
metal: 0.09%
pop: 0.10%
reggae: 91.65%
rock: 0.80%
--- Prediction Results ---
Predicted Genre: REGGAE
Confidence: 91.65%


In [6]:
import joblib

MLP = {
    "model": model,
    "scaler": scaler,
    "label_encoder": encoder
}
joblib.dump(MLP, f'../models/MLP_{NUMBER_OF_SECONDS}_sec.joblib')

['../models/MLP_3_sec.joblib']